# Spark + Scala + Apache Ozone

Этот блокнот выполняется ядром **Apache Toree (Scala)**. Spark уже настроен на Ozone: путь `ofs://om/spark/data/...` указывает на volume `spark`, bucket `data` и каталог внутри него.

Выполняйте ячейки сверху вниз. Они создадут DataFrame, запишут его в Ozone в формате Parquet и прочитают обратно.

In [ ]:
import org.apache.spark.sql.SparkSession

val ozoneSpark = SparkSession.builder()
  .appName("Jupyter Scala with Apache Ozone")
  .getOrCreate()

import ozoneSpark.implicits._

println(s"Spark ${ozoneSpark.version}")
println(ozoneSpark.sparkContext.hadoopConfiguration.get("fs.ofs.impl"))

## Запись DataFrame в Ozone

Обычный Scala `Seq` превращается в Spark DataFrame и сохраняется как Parquet.

In [ ]:
val ozonePath = "ofs://om/spark/data/notebook-users-scala"

val users = Seq(
  (1L, "Анна", "Москва"),
  (2L, "Борис", "Казань"),
  (3L, "Светлана", "Москва")
).toDF("id", "name", "city")

users.show(false)
users.write.mode("overwrite").parquet(ozonePath)

## Чтение из Ozone и преобразование DataFrame

In [ ]:
val fromOzone = ozoneSpark.read.parquet(ozonePath)

fromOzone.printSchema()
fromOzone.orderBy("id").show(false)
fromOzone.groupBy("city").count().orderBy("city").show(false)

require(fromOzone.count() == 3)
println("SCALA_OZONE_OK rows=3")

## Просмотр файлов средствами Hadoop FileSystem

Это показывает физические объекты Parquet, созданные Spark внутри bucket.

In [ ]:
import java.net.URI
import org.apache.hadoop.fs.{FileSystem, Path}

val ozoneFs = FileSystem.get(new URI("ofs://om/"), ozoneSpark.sparkContext.hadoopConfiguration)
ozoneFs.listStatus(new Path(ozonePath)).foreach { status =>
  println(s"${status.getPath.getName}\t${status.getLen} bytes")
}

## Iceberg-таблица в Ozone

Каталог `ozone` уже настроен, а его хранилище находится в `ofs://om/spark/warehouse`.

In [ ]:
ozoneSpark.sql("CREATE NAMESPACE IF NOT EXISTS ozone.notebook")
ozoneSpark.sql("""
  CREATE TABLE IF NOT EXISTS ozone.notebook.events (
    id BIGINT,
    message STRING
  ) USING iceberg
""")
ozoneSpark.sql("INSERT INTO ozone.notebook.events VALUES (1, 'Scala notebook works')")
ozoneSpark.sql("SELECT * FROM ozone.notebook.events ORDER BY id DESC LIMIT 10").show(false)